# VeloceAI — Colab Free (one-click launch)
1. اختَر GPU من Runtime → Change runtime type إذا كان متاحًا (اختياري؛ CPU أبطأ).
2. شغّل خلية التشغيل الوحيدة بالأسفل وانتظر ظهور رابط VeloceAI.
3. افتح الرابط واترك هذه الجلسة تعمل أثناء التحسين. الرابط مؤقت ويتوقف عند انتهاء جلسة Colab.

لا تحتاج Hugging Face أو اشتراكًا مدفوعًا. النموذج يُحمّل تلقائيًا عند أول عملية تحسين.

In [ ]:
import os, sys, time, re, pathlib, subprocess, urllib.request, json

def run(command):
    print('Running:', ' '.join(command[:5]), '...')
    subprocess.run(command, check=True)

# Install only the app dependencies; keep Colab's preinstalled PyTorch build.
run(['apt-get','-qq','update'])
run(['apt-get','-qq','install','-y','ffmpeg'])
run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn[standard]','python-multipart','pillow','opencv-python-headless','spandrel'])

base='https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/'
urllib.request.urlretrieve(base+'colab_server.py','/content/colab_server.py')
urllib.request.urlretrieve(base+'index.html','/content/index.html')

# Stop old processes if this launch cell is run again.
for name in ('tunnel','server'):
    old=globals().get(name)
    try:
        if old and old.poll() is None:
            old.terminate(); old.wait(timeout=8)
    except Exception:
        pass

server_log=open('/content/veloceai-server.log','w')
server=subprocess.Popen([sys.executable,'/content/colab_server.py'],stdout=server_log,stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        health=json.loads(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=3).read())
        if health.get('ok'): break
    except Exception: pass
    time.sleep(2)
else:
    print(pathlib.Path('/content/veloceai-server.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('VeloceAI server did not start. See the log above.')

cloudflared='/usr/local/bin/cloudflared'
if not pathlib.Path(cloudflared).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    os.chmod(cloudflared,0o755)
tunnel_log=open('/content/veloceai-tunnel.log','w')
tunnel=subprocess.Popen([cloudflared,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(60):
    time.sleep(2)
    log=pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
    if match:
        public_url=match.group(0); break
if not public_url:
    print(pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('Could not obtain a temporary website URL.')

print('\n'+'='*60)
print('VeloceAI website:',public_url)
print('Health:',urllib.request.urlopen(public_url+'/health',timeout=20).read().decode())
print('Keep this Colab cell/session running while using the website.')
print('='*60)
